## Creates support tables

Tables created in this script
- verb_matches
- pattern_support

In [1]:
import sqlite3
import pandas as pd
import sys
sys.path.append("..")
from common_display import display_db_table 

### Configuration

In [2]:
DB_DIR = "../example_data"

PATTERN_DB = f"{DB_DIR}/verb_patterns.db"
TRANSACTION_DB = f"{DB_DIR}/transactions.db"

PATTERNS_TABLE = "patterns"
SEMANTIC_ANNOTATIONS = "semantic_annotations"
VERB_MATCHES_TABLE = "verb_matches"
PATTERNS_META_TABLE = "patterns_meta"
PATTERN_SUPPORT_TABLE = "pattern_support"

## Connect to db

In [3]:
con = sqlite3.connect(PATTERN_DB)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" as trans ')

## Workflow

### I tabel verb_matches

In [4]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=VERB_MATCHES_TABLE))

cur.execute("""DROP TABLE IF EXISTS verb_matches_1""")

cur.execute("""
CREATE TABLE verb_matches_1 AS
SELECT DISTINCT
    pat_id,
    head_id
FROM
(
    SELECT 
        pat_id, 
        tr_head.id as head_id
    FROM
        {tbl} as pat
    INNER JOIN
        trans.transaction_head as tr_head
    ON
        pat.verb_word = tr_head.verb
    AND pat.verb_compound = tr_head.verb_compound
)
""".format(tbl=PATTERNS_TABLE))

cur.execute(
"""
CREATE TABLE {tbl1} AS 
Select distinct
    vm.pat_id,
    vm.head_id,
    sem.phrase_nr,
    sem.semantic_role,
    sem.certainty
from 
verb_matches_1 as vm
join 
{tbl2} as sem
on vm.pat_id = sem.pattern_id 
""".format(tbl1=VERB_MATCHES_TABLE, tbl2=SEMANTIC_ANNOTATIONS)
)

cur.execute("""DROP TABLE IF EXISTS verb_matches_1""")

cur.execute("""CREATE INDEX v_match_pat_id_idx ON {tbl}(pat_id)""".format(tbl=VERB_MATCHES_TABLE))
cur.execute("""CREATE INDEX v_match_head_id_idx ON {tbl}(head_id)""".format(tbl=VERB_MATCHES_TABLE))

con.commit()

CPU times: user 78 ms, sys: 4.61 ms, total: 82.6 ms
Wall time: 101 ms


In [5]:
display_db_table(con, VERB_MATCHES_TABLE)

,pat_id,head_id,phrase_nr,semantic_role,certainty
0,1,44,1,isik,vahel
1,1,44,1,koht,vahel
2,1,44,1,muu,mitte kunagi
3,1,54,1,isik,vahel
4,1,54,1,koht,vahel


### II tabel pattern_support

In [7]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=PATTERN_SUPPORT_TABLE))

cur.execute("""
CREATE TABLE {new_table} AS
SELECT
    pm.pat_id,
    verb_word,
    verb_compound,
    phrase_case,
    verb_match_count AS verb_occurrence_count,
    phrase_count AS absolute_support,
    CAST(phrase_count AS REAL) / CAST(verb_match_count AS REAL) * 100 AS relative_support
FROM
(
    SELECT 
        pat.pat_id as pat_id,
        verb_word,
        verb_compound,
        phrase_case,
        count(*) AS verb_match_count
    FROM
        {tbl1} as vm
    INNER JOIN
        {tbl2} as pat
    ON
        pat.pat_id = vm.pat_id
    GROUP BY
        pat.pat_id
) as tbl
INNER JOIN
    {tbl3} as pm
ON
    tbl.pat_id = pm.pat_id
ORDER BY
    relative_support DESC
""".format(new_table=PATTERN_SUPPORT_TABLE, tbl1=VERB_MATCHES_TABLE, tbl2=PATTERNS_TABLE, tbl3=PATTERNS_META_TABLE))

CPU times: user 16.8 ms, sys: 411 µs, total: 17.2 ms
Wall time: 20.2 ms


In [8]:
display_db_table(con, PATTERN_SUPPORT_TABLE)

,pat_id,verb_word,verb_compound,phrase_case,verb_occurrence_count,absolute_support,relative_support
0,2,tulema,,abl,60,20,33.333333
1,4,nõudma,,abl,3,1,33.333333
2,5,võtma,,abl,6,2,33.333333
3,6,ootama,,abl,9,3,33.333333
4,7,leidma,,abl,18,6,33.333333


In [9]:
con.close()